In [1]:
!pip install pyspark

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, when, count, isnan, to_date

spark = SparkSession.builder \
    .appName("SalesDataLake_Bronze_to_Silver") \
    .getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"
df_bronze = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("Dataset Schema:")
df_bronze.printSchema()
print("First 5 rows:")
df_bronze.show(5)

original_count = df_bronze.count()
print(f"Total Original Records: {original_count}")

duplicate_count = original_count - df_bronze.dropDuplicates().count()
print(f"Number of exact duplicate rows: {duplicate_count}")

print("Count of Null/NaN values per column:")
null_exprs = []
for c, dtype in df_bronze.dtypes:
    if dtype in ('double', 'float'):
        null_exprs.append(count(when(col(c).isNull() | isnan(col(c)), c)).alias(c))
    else:
        null_exprs.append(count(when(col(c).isNull(), c)).alias(c))

df_bronze.select(null_exprs).show()

Dataset Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First 5 rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+------------

In [15]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType
from datetime import datetime

print("--- STARTING REFINED PREPROCESSING PIPELINE ---")
initial_count = df_bronze.count()
print(f"Initial row count: {initial_count}\n")

df_silver = df_bronze.dropDuplicates()
print(f"[Step 1] Exact duplicates removed. Current count: {df_silver.count()}")

if "order_id" in df_silver.columns:
    df_silver = df_silver.dropDuplicates(subset=["order_id"])
print(f"[Step 2] Structural duplicates removed. Current count: {df_silver.count()}")

df_silver = df_silver.dropna(subset=["order_id", "customer_id"])
print(f"[Step 3] Rows with missing critical IDs dropped. Current count: {df_silver.count()}\n")

text_columns = [f.name for f in df_silver.schema.fields if f.dataType.typeName() == 'string']
for c in text_columns:
    df_silver = df_silver.withColumn(c, F.initcap(F.trim(F.regexp_replace(F.col(c), r'\s+', ' '))))
print("[Step 4] String normalization applied. Sample text data:")
df_silver.select("customer_name", "category", "city").show(3, truncate=False)

medians = df_silver.select(
    F.percentile_approx("quantity", 0.5).alias("q_med"),
    F.percentile_approx("unit_price", 0.5).alias("u_med"),
    F.percentile_approx("discount_percent", 0.5).alias("d_med")
).collect()[0]

print(f"[Step 5] Calculated Medians -> Quantity: {medians['q_med'] or 1}, Price: {medians['u_med'] or 0}, Discount: {medians['d_med'] or 0}")

df_silver = df_silver.na.fill({
    "quantity": medians["q_med"] or 1,
    "unit_price": medians["u_med"] or 0,
    "discount_percent": medians["d_med"] or 0,
    "category": "Uncategorized",
    "payment_method": "Unknown",
    "order_status": "Unknown",
    "city": "Unknown",
    "state": "Unknown"
})
print("[Step 5] Nulls imputed. Showing sample of normally nullable columns:")
df_silver.select("quantity", "unit_price", "category", "city").show(3)

df_silver = df_silver.filter(
    (F.col("quantity") > 0) &
    (F.col("unit_price") >= 0) &
    (F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100)
)
print(f"[Step 6] Negative/Invalid numerical bounds filtered. Current count: {df_silver.count()}\n")

quantiles = df_silver.approxQuantile("quantity", [0.25, 0.75], 0.01)
Q1, Q3 = quantiles[0], quantiles[1]
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

df_silver = df_silver.withColumn(
    "quantity",
    F.when(F.col("quantity") > upper_bound, F.lit(int(upper_bound))).otherwise(F.col("quantity"))
)
print(f"[Step 7] Outliers capped. Upper bound for quantity set to: {int(upper_bound)}")

valid_statuses = ["Delivered", "Shipped", "Cancelled", "Returned", "Processing", "Unknown"]
df_silver = df_silver.withColumn(
    "order_status",
    F.when(F.col("order_status").isin(valid_statuses), F.col("order_status")).otherwise(F.lit("Unknown"))
)
print("[Step 8] Categorical enforcement applied. Status distribution:")
df_silver.groupBy("order_status").count().show()

def safe_parse_date(date_str):
    if not date_str:
        return None
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d']
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).date()
        except ValueError:
            continue
    return None

safe_date_udf = F.udf(safe_parse_date, DateType())
df_silver = df_silver.withColumn("order_date", safe_date_udf(F.col("order_date")))

df_silver = df_silver.filter(
    F.col("order_date").isNotNull() &
    (F.col("order_date") <= F.lit(datetime.today().date()))
)
print(f"[Step 9] Valid dates parsed. Future/unparseable dates dropped. Current count: {df_silver.count()}\n")

df_silver = df_silver.withColumn(
    "total_transaction_value",
    F.round((F.col("quantity") * F.col("unit_price")) * (1 - F.col("discount_percent") / 100), 2)
)
print("[Step 10] Feature engineering complete. Added 'total_transaction_value'.")
df_silver.select("quantity", "unit_price", "discount_percent", "total_transaction_value").show(5)

print(f"--- PIPELINE COMPLETE: Final Row Count is {df_silver.count()} ---")

--- STARTING REFINED PREPROCESSING PIPELINE ---
Initial row count: 1000

[Step 1] Exact duplicates removed. Current count: 990
[Step 2] Structural duplicates removed. Current count: 982
[Step 3] Rows with missing critical IDs dropped. Current count: 982

[Step 4] String normalization applied. Sample text data:
+--------------+--------+-------+
|customer_name |category|city   |
+--------------+--------+-------+
|Keerthi Mishra|Clothing|Mumbai |
|Arjun Singh   |Clothing|Chennai|
|Keerthi Rao   |Clothing|Pune   |
+--------------+--------+-------+
only showing top 3 rows
[Step 5] Calculated Medians -> Quantity: 5, Price: 1830, Discount: 20
[Step 5] Nulls imputed. Showing sample of normally nullable columns:
+--------+----------+--------+-------+
|quantity|unit_price|category|   city|
+--------+----------+--------+-------+
|       2|      6710|Clothing| Mumbai|
|       6|       680|Clothing|Chennai|
|       6|      3080|Clothing|   Pune|
+--------+----------+--------+-------+
only showing t

In [16]:
import os
import glob
import shutil

final_count = df_silver.count()
print(f"Row count BEFORE preprocessing: {original_count}")
print(f"Row count AFTER preprocessing: {final_count}")

temp_silver_dir = "data_lake/silver/temp_sales"
df_silver.coalesce(1).write.mode("overwrite").csv(temp_silver_dir, header=True)

part_file = glob.glob(f"{temp_silver_dir}/part-*.csv")[0]
final_silver_path = "data_lake/silver/sales_silver.csv"
os.rename(part_file, final_silver_path)

shutil.rmtree(temp_silver_dir)
print(f"Successfully saved as exactly: {final_silver_path}")
df_silver_read = spark.read.csv(final_silver_path, header=True, inferSchema=True)
print("\nVerifying saved Silver Data:")
df_silver_read.show(5)

print("""
Preprocessing Summary:
1. Removed exact duplicate rows.
2. Handled missing values by dropping rows without IDs and filling missing numerics with 0.
3. Trimmed trailing/leading spaces and standardized capitalization for all text fields.
4. Filtered out logically invalid numerical values (e.g., negative prices or quantities).
5. Standardized the date column and removed unparseable date values.
""")

Row count BEFORE preprocessing: 1000
Row count AFTER preprocessing: 953
Successfully saved as exactly: data_lake/silver/sales_silver.csv

Verifying saved Silver Data:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+-----------------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|total_transaction_value|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+-----------------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|                10736.0|
|  100002|      C0042|   Arjun Singh|      T-shirt|Clothing|       6|       680|     